In [1]:
import Pkg
Pkg.activate(".")
Pkg.instantiate()

  Activating project at `~/repos/WaveGreen2D/notebooks`


In [2]:
using StaticArrays
using SpecialFunctions

In [11]:
function F₀(t::SVector{2, Float64})
    X, V₃ = t
    
    Zₐ = sqrt(V₃^2 + X^2)
    
    F = F₁(t) + 2*log(Zₐ)

    return F
end


function ∇F₀(t::SVector{2, Float64}, ∇t::SVector{2, Float64})
    X, V₃ = t
    ∇X, ∇V₃ = ∇t

    X² = X^2
    V₃² = V₃^2
    
    Zₐ² = V₃² + X²
    Zₐ = sqrt(Zₐ²)
    
    g = log(Zₐ)
    gˣ = X * ∇X / Zₐ²
    gᶻ = V₃ * ∇V₃ / Zₐ²
    ∇g = SVector{2, Float64}(gˣ, gᶻ)

    F1, ∇F1 = ∇F₁(t, ∇t)
    
    F = F1 + 2*g
    ∇F = ∇F1 + 2*∇g

    return F, ∇F
end


function HF₀(t::SVector{2, Float64}, ∇t::SVector{2, Float64})
    X, V₃ = t
    ∇X, ∇V₃ = ∇t

    X² = X^2
    V₃² = V₃^2
    
    Zₐ² = V₃² + X²
    Zₐ = sqrt(Zₐ²)
    Zₐ⁴ = Zₐ² * Zₐ²
    
    g = log(Zₐ)
    gˣ = X * ∇X / Zₐ²
    gᶻ = V₃ * ∇V₃ / Zₐ²
    ∇g = SVector{2, Float64}(gˣ, gᶻ)
    gˣˣ = (V₃² - X²) * ∇X^2 / Zₐ⁴
    gˣᶻ = -2 * X * V₃ * ∇X * ∇V₃ / Zₐ⁴
    gᶻᶻ = (X² - V₃²) * ∇V₃^2 / Zₐ⁴
    Hg = SMatrix{2, 2, Float64}([gˣˣ gˣᶻ; gˣᶻ gᶻᶻ])

    F1, ∇F1, HF1 = HF₁(t, ∇t)
    
    F = F1 + 2*g
    ∇F = ∇F1 + 2*∇g
    HF = HF1 + 2*Hg

    return F, ∇F, HF
end


function F₁(t::SVector{2, Float64})
    X, V₃ = t
    Z = V₃ - im*X
    
    e = SpecialFunctions.expintx(-Z)
    f = exp(-V₃) * sin(X)
    
    F = 2 * (real(e) - π * f)

    return F
end


function ∇F₁(t::SVector{2, Float64}, ∇t::SVector{2, Float64})
    X, V₃ = t
    ∇X, ∇V₃ = ∇t
    
    Z = V₃ - im*X
    ∇Z = SVector{2, ComplexF64}(-im*∇X, ∇V₃)
    
    e = SpecialFunctions.expintx(-Z)
    e′ = -e - 1/Z
    ∇e = e′ .* ∇Z
    
    e⁻ⱽ³ = exp(-V₃)
    f = e⁻ⱽ³ * sin(X)
    fˣ = e⁻ⱽ³ * cos(X) * ∇X
    fᶻ = -f * ∇V₃
    ∇f = SVector{2, Float64}(fˣ, fᶻ)
    
    F = 2 * (real(e) - π * f)
    ∇F = 2 .* (real(∇e) .- π .* ∇f)

    return F, ∇F
end


function HF₁(t::SVector{2, Float64}, ∇t::SVector{2, Float64})
    X, V₃ = t
    ∇X, ∇V₃ = ∇t
    
    Z = V₃ - im*X
    ∇Z = SVector{2, ComplexF64}(-im*∇X, ∇V₃)
    
    e = SpecialFunctions.expintx(-Z)
    e′ = -e - 1/Z
    e′′ = -e′ + 1/Z^2
    ∇e = e′ .* ∇Z
    He = e′′ .* ∇Z * transpose(∇Z)
    
    e⁻ⱽ³ = exp(-V₃)
    f = e⁻ⱽ³ * sin(X)
    fˣ = e⁻ⱽ³ * cos(X) * ∇X
    fᶻ = -f * ∇V₃
    fˣˣ = -f * ∇X^2
    fˣᶻ = -fˣ * ∇V₃
    fᶻᶻ = f * ∇V₃^2
    ∇f = SVector{2, Float64}(fˣ, fᶻ)
    Hf = SMatrix{2, 2, Float64}([fˣˣ fˣᶻ; fˣᶻ fᶻᶻ])

    F = 2 * (real(e) - π * f)
    ∇F = 2 .* (real(∇e) .- π .* ∇f)
    HF = 2 .* (real(He) .- π .* Hf)
    
    return F, ∇F, HF
end

HF₁ (generic function with 1 method)

In [12]:
ω = 0.8
g = 9.81
K = ω^2 / g

x, z = 1.0, -1.0
ξ, ζ = -2.0, -1.5

R̄ = x - ξ
R = abs(R̄)
R² = R^2
∇R = sign(R̄)

v₃ = -z - ζ

X = K * R
∇X = K*∇R

V₃ = K * v₃
∇V₃ = -K

t = SA[X, V₃]
∇t = SA[∇X, ∇V₃];

In [13]:
F₀(t)

-2.042976644141568

In [14]:
F, ∇F = ∇F₀(t, ∇t)

println(F)
println(∇F)

-2.042976644141568
[-0.13191186280673683, 0.04513419646671696]


In [15]:
F, ∇F, HF = HF₀(t, ∇t)

println(F)
println(∇F)
println(HF)

-2.042976644141568
[-0.13191186280673683, 0.04513419646671696]
[-0.024334551611872507 -0.03427389101782222; -0.03427389101782222 0.024334551611872507]


In [8]:
F₁(t)

0.6918226052163938

In [9]:
F, ∇F = ∇F₁(t, ∇t)

println(F)
println(∇F)

0.6918226052163938
[-0.5253544857575565, 0.37300304892573327]


In [10]:
F, ∇F, HF = HF₁(t, ∇t)

println(F)
println(∇F)
println(HF)

0.6918226052163938
[-0.5253544857575565, 0.37300304892573327]
[-0.0006849950410581984 -0.16327147231317293; -0.16327147231317293 0.0006849950410581984]
